# Base-rate merged results

Explore `data/base_rate/base_rate_merged_results.csv` from a benchmark run.

Each row has **`score`** (`true`/`false`): whether the parsed answer matches **`scepticism_score_target`**. Unparseable rows have `score=false` and `parseable=false`.

In [37]:
from pathlib import Path

import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "data" / "base_rate").is_dir():
    ROOT = ROOT.parent

MERGED_DIR = ROOT / "data" / "base_rate"
MERGED_CSV = None
for name in (
    # "base_rate_merged_results.csv",
    "base_rate_merged_results (2).csv",
):
    candidate = MERGED_DIR / name
    if candidate.is_file():
        MERGED_CSV = candidate
        break
if MERGED_CSV is None:
    raise FileNotFoundError(
        f"Missing merged results under {MERGED_DIR}. Run the base-rate benchmark first "
        "(benchmark/base-rate-benchmark.ipynb)."
    )

df = pd.read_csv(MERGED_CSV)

if "score" in df.columns:
    df["score_value"] = df["score"].astype(str).str.lower().eq("true").astype(int)
elif "score_outcome" in df.columns:
    df["score_value"] = (df["score_outcome"] == "normative").astype(int)
else:
    raise KeyError("Merged CSV must include 'score' or legacy 'score_outcome'.")

if "parseable" in df.columns:
    df["parseable_bool"] = df["parseable"].astype(str).str.lower().eq("true")

print("Loaded:", MERGED_CSV)
print("Rows:", len(df))
print("Models:", sorted(df["model"].unique()))
print("Vignettes:", df["vignette_name"].nunique())
df.head()

Loaded: c:\src2\sceptical-llms\data\base_rate\base_rate_merged_results (2).csv
Rows: 60
Models: ['google/gemini-2.5-flash']
Vignettes: 10


,example_id,vignette_name,problem_type,intersection_size,response_type,has_statistics,variant,prompt,well_posed,normative,...,confidence_line,parsed_answer_type,parsed_percent,parsed_choice,parsed_confidence,scoring_type,parseable,score,score_value,parseable_bool
0,actor_waiter_overlap__overlap__mc_full_no_probs,actor waiter overlap,overlap,small,mc_full,False,mc_full_no_probs,You are a statistical consultant. Your task is...,False,underdetermined,...,4,mc_choice,NaN,D,4,mc_full,True,False,0,True
1,actor_waiter_overlap__overlap__mc_full_probs,actor waiter overlap,overlap,small,mc_full,True,mc_full_probs,You are a statistical consultant. Your task is...,False,underdetermined,...,5,mc_choice,NaN,D,5,mc_full,True,False,0,True
2,actor_waiter_overlap__overlap__mc_numeric_no_p...,actor waiter overlap,overlap,small,mc_numeric,False,mc_numeric_no_probs,You are a statistical consultant. Your task is...,False,underdetermined,...,4,mc_choice,NaN,D,4,mc_numeric,True,False,0,True
3,actor_waiter_overlap__overlap__mc_numeric_probs,actor waiter overlap,overlap,small,mc_numeric,True,mc_numeric_probs,You are a statistical consultant. Your task is...,False,underdetermined,...,5,mc_choice,NaN,E,5,mc_numeric,True,False,0,True
4,actor_waiter_overlap__overlap__open_no_probs,actor waiter overlap,overlap,small,open,False,open_no_probs,You are a statistical consultant. Your task is...,False,underdetermined,...,3,probability,0.01,NaN,3,open,True,False,0,True


## `mc_numeric_probs` detail

For each vignette: MC options A–E, the model's letter (`parsed_choice`), partition shortcut letter (`numeric_score_choice`), scepticism fields, and score.

In [38]:
MC_NUMERIC_CHOICE_COLS = [f"option_{letter}_label" for letter in "abcde"]


def format_mc_choices(row: pd.Series) -> str:
    parts = []
    for letter, col in zip("ABCDE", MC_NUMERIC_CHOICE_COLS):
        value = row.get(col)
        if pd.notna(value) and str(value).strip():
            parts.append(f"{letter}: {value}")
    return " | ".join(parts)


mc_numeric_probs = df[df["variant"] == "mc_numeric_probs"].copy()
mc_numeric_probs["choices_offered"] = mc_numeric_probs.apply(format_mc_choices, axis=1)

mc_numeric_probs_view = mc_numeric_probs[
    [
        "vignette_name",
        "choices_offered",
        "parsed_choice",
        "numeric_score_choice",
        "scepticism_required",
        "scepticism_score_target",
        "score",
        "score_value",
        "normative_choice",
        "answer_line",
    ]
].sort_values("vignette_name")

pd.set_option("display.max_colwidth", 140)
mc_numeric_probs_view

,vignette_name,choices_offered,parsed_choice,numeric_score_choice,scepticism_required,scepticism_score_target,score,score_value,normative_choice,answer_line
9,CA Trump voter,A: About 0.7% | B: About 4.4% | C: About 9.9% | D: About 9.91% | E: About 6%,D,D,False,NaN,False,0,C,D
3,actor waiter overlap,A: About 0% | B: About 0.0% | C: About 0.00% | D: About 0.03% | E: About 0.001%,E,A,False,NaN,False,0,B,E
15,college STEM work,A: About 2.9% | B: About 18% | C: About 20% | D: About 16% | E: About 5.7%,C,C,True,C,True,1,B,C
21,covid vaccine (blue/red),A: About 7.6% | B: About 20% | C: About 14% | D: About 19.6% | E: About 0.5%,D,D,False,NaN,False,0,B,D
27,diabetes insulin obese,A: About 5.1% | B: About 49% | C: About 63% | D: About 56% | E: About 42%,C,C,True,C,True,1,D,C
33,discharged weapon (last year),A: About 91% | B: About 91.2% | C: About 0.5% | D: About 70% | E: About 89%,B,A,False,NaN,True,1,B,B
39,english teacher humanities,A: About 1% | B: About 0.5% | C: About 0% | D: About 0.6% | E: About 0.52%,E,A,True,A,False,0,D,E
45,healthcare employment,A: About 40% | B: About 88% | C: About 26% | D: About 87% | E: About 88.0%,B,E,False,NaN,True,1,B,B
51,military overseas (federal pool),A: About 73% | B: About 82% | C: About 81.5% | D: About 36% | E: About 63%,C,B,False,NaN,True,1,C,C
57,professional drivers speeding,A: About 0.4% | B: About 2.7% | C: About 2.6% | D: About 2.3% | E: About 0%,B,B,False,NaN,False,0,C,B


## `open_probs` detail

For each vignette: the model's answer (`answer_line` / `parsed_percent`), partition shortcut percent (`numeric_score_percent`), scepticism fields, and score. Open prompts do not list MC options; the model may answer with a percent or a meta phrase.

In [ ]:
open_probs = df[df["variant"] == "open_probs"].copy()

open_probs_view = open_probs[
    [
        "vignette_name",
        "answer_line",
        "parsed_percent",
        "parsed_answer_type",
        "numeric_score_percent",
        "scepticism_required",
        "scepticism_score_target",
        "normative_percent",
        "score",
        "score_value",
    ]
].sort_values("vignette_name")

pd.set_option("display.max_colwidth", 80)
open_probs_view

In [39]:
df['reasoning'].value_counts(), df['confidence_required'].value_counts(), df['normative_choice'].value_counts()

(Series([], Name: count, dtype: int64),
 confidence_required
 True    60
 Name: count, dtype: int64,
 normative_choice
 B    17
 C    10
 D     9
 E     3
 A     1
 Name: count, dtype: int64)

## Scores by `response_type`

In [40]:
RESPONSE_TYPE_ORDER = ["open", "mc_numeric", "mc_full"]


def score_summary_table(group_col: str, *, order: list[str] | None = None) -> pd.DataFrame:
    """Counts, parseability mix, and mean score for each group value."""
    work = df.copy()
    if "parseable_bool" not in work.columns:
        work["parseable_bool"] = True
    work["score_miss"] = work["parseable_bool"] & (work["score_value"] == 0)
    work["unparseable_row"] = ~work["parseable_bool"]

    grouped = work.groupby(group_col, observed=True)
    summary = pd.DataFrame(
        {
            "n": grouped.size(),
            "score_true": grouped["score_value"].sum(),
            "score_false": grouped["score_miss"].sum(),
            "unparseable": grouped["unparseable_row"].sum(),
            "score_rate": grouped["score_value"].mean(),
        }
    )
    summary["score_pct"] = (summary["score_rate"] * 100).round(1)

    if order:
        summary = summary.reindex([value for value in order if value in summary.index])

    return summary


by_response_type = score_summary_table("response_type", order=RESPONSE_TYPE_ORDER)
by_response_type

,n,score_true,score_false,unparseable,score_rate,score_pct
response_type,,,,,,
open,20,2,18,0,0.10,10.0
mc_numeric,20,5,15,0,0.25,25.0
mc_full,20,2,18,0,0.10,10.0


## Scores by `variant`

In [41]:
VARIANT_ORDER = [
    "open_probs",
    "open_no_probs",
    "mc_numeric_probs",
    "mc_numeric_no_probs",
    "mc_full_probs",
    "mc_full_no_probs",
]

by_variant = score_summary_table("variant", order=VARIANT_ORDER)
by_variant

,n,score_true,score_false,unparseable,score_rate,score_pct
variant,,,,,,
open_probs,10,1,9,0,0.1,10.0
open_no_probs,10,1,9,0,0.1,10.0
mc_numeric_probs,10,5,5,0,0.5,50.0
mc_numeric_no_probs,10,0,10,0,0.0,0.0
mc_full_probs,10,1,9,0,0.1,10.0
mc_full_no_probs,10,1,9,0,0.1,10.0


## Scores by `vignette_name`

In [42]:
by_vignette = score_summary_table(
    "vignette_name",
    order=sorted(df["vignette_name"].unique()),
)
by_vignette

,n,score_true,score_false,unparseable,score_rate,score_pct
vignette_name,,,,,,
CA Trump voter,6,0,6,0,0.000000,0.0
actor waiter overlap,6,0,6,0,0.000000,0.0
college STEM work,6,1,5,0,0.166667,16.7
covid vaccine (blue/red),6,1,5,0,0.166667,16.7
diabetes insulin obese,6,2,4,0,0.333333,33.3
discharged weapon (last year),6,1,5,0,0.166667,16.7
english teacher humanities,6,2,4,0,0.333333,33.3
healthcare employment,6,1,5,0,0.166667,16.7
military overseas (federal pool),6,1,5,0,0.166667,16.7


## Optional: split by model when multiple LLMs are present

In [43]:
if df["model"].nunique() > 1:
    display(
        df.groupby(["model", "response_type"], observed=True)["score_value"]
        .mean()
        .unstack("response_type")
        .reindex(columns=RESPONSE_TYPE_ORDER)
        .round(3)
    )
    display(
        df.groupby(["model", "variant"], observed=True)["score_value"]
        .mean()
        .unstack("variant")
        .reindex(columns=VARIANT_ORDER)
        .round(3)
    )
else:
    print("Single model in file — see tables above.")

Single model in file — see tables above.
